# Các lớp

## Drawer

In [82]:
import cv2
import numpy as np

class Draw:
    color = {
        1 : (255, 0, 0)
    }
    def Convertxyxy(self, data):
        x_1 = data[0] - data[2] / 2
        y_1 = data[1] - data[3] / 2
        x_2 = data[0] + data[2] / 2
        y_2 = data[1] + data[3] / 2

        return list(map(int, [x_1, y_1, x_2, y_2]))
    
    def DrawFromModel(self, img, results):
        for r in results[0].boxes:
            x_1, y_1, x_2, y_2 = map(int, r.xyxy[0])
            cv2.rectangle(img, (x_1, y_1), (x_2, y_2), color = (0, 255, 0), thickness=2)

        return img   

    def DrawFromMap(self, img, chars):
        # [cls_id, ft_color_1, ft_color_2, ft_color_3, X_cen, Y_cen, w, h, <speed>]
        for id in range(len(chars)):
            char = chars[id]
            if char[-2] == 0:
                cls_id = char[0]
                box = self.Convertxyxy(char[4:8])

                name = f'{cls_id}-{id}'

                point_1 = list(map(int,char[4:6]))
                point_2 = [int(p_1 - p_2) for p_1, p_2 in zip(char[4:6], char[8:10])]

                cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), color = Draw.color.get(cls_id, (0, 255, 0)), thickness=2)
                cv2.line(img, point_1, point_2, (255, 255, 255), 3)
                # cv2.putText(img, name, (box[0], box[1] - 10),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        return img

## Character

In [83]:
import numpy as np
import math

class Character:
  # def __init__(self) -> None:
  #   self.cls_id = -1
  #   self.feature_color = [-1, -1, -1] # Use Mean HSV
  #   self.feature_point = [-1, -1, -1] # [X_center, Y_center, w, h]
  #   self.speed = [0, 0, 0, 0] # [V_x, V_y, V_w, V_h]
  #   self.lost = 0
  #   self.violate = 0

  def Read_Char(self, input: list) -> None:
    # input = [cls_id, <ft_color>, <ft_point>, <speed>, lost, violate]
    self.cls_id = int(input[0])
    self.feature_color = input[1:4]
    self.feature_point = input[4:8]
    self.speed = input[8:12]
    self.lost = input[12]
    self.violate = input[13]

  def Check_Lost(self, num_lost: int) -> bool:
    return True if self.lost >= num_lost + 1 else False

  def Get_Cls(self) -> int:
    return self.cls_id
  
  @staticmethod
  def Get_Cos(angle_deg) -> int:
    if angle_deg == 90 or angle_deg == 270:
      return 0
    return np.cos(np.radians(angle_deg))
  
  @staticmethod
  def Get_Sin(angle_deg) -> int:
    if angle_deg == 0 or angle_deg == 180:
      return 0
    return np.sin(np.radians(angle_deg))

  def Get_Direction(self) -> list[float]:
    speed = self.speed[:2]
    dis = np.sqrt(speed[0]**2 + speed[1]**2)
    if dis <= 3:
      return [0, 0]
    deg = math.degrees(math.atan2(speed[1], speed[0])) % 360
    deg = int(round(deg, 0))
    return [Character.Get_Cos(deg), Character.Get_Sin(deg)]


  def Get_Point_Future(self, lecay: float) -> list:
    temp = np.array(self.feature_point)
    temp += (self.lost + 1) * lecay * np.array(self.speed)
    return temp.tolist()

  def Get_ID_Check(self):
    return int((self.feature_point[0] - self.speed[0]) / 53.25), int((self.feature_point[1] - self.speed[1]) / 30)

  def Data_For_reID(self) -> list:
    ft_point = self.Get_Point_Future(0.9)
    return [self.cls_id] + self.feature_color + ft_point + self.speed + [self.lost, self.violate]

  '''
  input = [cls_id, ft_color_1, ft_color_2, ft_color_3, X_cen, Y_cen, w, h]
  '''
  @staticmethod
  def New_Char(input: list) -> list:
    return input + [0, 0, 0, 0, 0, 0] # thêm speed, lost và violate

## Checker

In [84]:
import numpy as np
from collections import Counter
# from worker.character import Character

class Checker:
  def __init__(self) -> None:
    self.active = False
    self.img_ids = []
    self.angles = []
    self.vehicle = []
    self.char_reader = Character()
    
  def Read_Checker(self, check: list) -> None:
    # check = [True, [...], [...], [...]]
    self.active = check[0]
    self.img_ids = check[1]
    self.angles = check[2]
    self.vehicle = check[3]

  def Save(self):
    result = []
    result.append(self.active)
    result.append(self.img_ids)
    result.append(self.angles)
    result.append(self.vehicle)
    return result
  
  #--------------------method_active  
  @staticmethod
  def Check_Direct(direct_1, direct_2, value = 0.35, type = 'Check'):
    x_1 = direct_1[0]
    y_1 = direct_1[1]
    x_2 = direct_2[0]
    y_2 = direct_2[1]

    if type == 'Check':
      dis = np.sqrt((x_1 - x_2)**2 + (y_1 - y_2)**2)
      return True if dis <= value else False
    if type == 'Avg':
      x_1 = x_1*0.9 + x_2*0.1
      y_1 = y_1*0.9 + y_2*0.1
      return [x_1, y_1]
  @staticmethod
  def Define_Veh(vehs: list) -> list:
    if not vehs:
        return []
    count = Counter(vehs)
    total = len(vehs)
    ratio_dict = {veh: freq / total for veh, freq in count.items()}
    max_ratio = max(ratio_dict.values())
    result = [veh for veh, ratio in ratio_dict.items() if ratio / max_ratio >= 0.6]
    return result

  def Active_False(self, img_id, char: list) -> list:
    self.char_reader.Read_Char(char)
    cls_id = self.char_reader.Get_Cls()
    char_angle = self.char_reader.Get_Direction()
    if np.sum(char_angle) == 0:
      return char
    if len(self.angles) == 0:
      self.angles.append(char_angle)
      self.vehicle.append([cls_id])
    else:
      if Checker.Check_Direct(self.angles[-1], char_angle):
        self.angles[-1] = Checker.Check_Direct(self.angles[-1], char_angle, type = 'Avg')
        self.vehicle[-1].append(cls_id)
      else:
        self.img_ids.append(img_id)
        if Checker.Check_Direct(self.angles[0], char_angle):
          self.active = True
          temp = self.img_ids[-1] if len(self.img_ids) > 0 else 0
          if img_id - temp <= 5:
            # Trường hợp xe đi ngược chiều
            self.active = False
            self.img_ids.pop()
            self.img_ids.pop()
            self.angles.pop()
            self.vehicle.pop()
        else:
          # Trường hợp đường trên 3 hướng
          self.angles.append(char_angle)
          self.vehicle.append([cls_id])
    if self.active:
      for i in range(len(self.vehicle)):
        self.vehicle[i] = Checker.Define_Veh(self.vehicle[i])
    if len(self.img_ids) == 0 and img_id >= 120:
      # Trường hợp đường chỉ có 1 hướng
      self.active = True
    if len(self.img_ids) != 0 and img_id - self.img_ids[-1] >= 120:
      self.active = True
    return char
  
  def Active_True(self, img_id, char: list) -> list:
    if self.Checker_Error(img_id % self.img_ids[-1], char):
      char[-1] = 1
    return char
  
  def Run(self, img_id, char: list) -> list:
    if self.active:
      return self.Active_True(img_id, char)
    else:
      return self.Active_False(img_id, char)

  #----------------Check_Error
  def Get_Code_Id(self, img_id):
    img_id = img_id % self.img_ids[-1]
    temp = 0
    for id in self.img_ids:
      if img_id > id:
        temp += 1
      else:
        break
    return temp
  
  def Checker_Error(self, img_id, char: list) -> list:
    self.char_reader.Read_Char(char)
    img_id = self.Get_Code_Id(img_id)

    if self.Wrong_Lane(id) or self.Wrong_Trend(id):
      return True
    return False
  
  def Wrong_Lane(self, id) -> bool:
    # Lỗi lấn làn
    if self.char_reader.Get_Cls() not in self.vehicle[id]:
      return True
    return False
  def Wrong_Trend(self, id) -> bool:
    # Lỗi ngược chiều, vượt đèn đỏ
    char_angle = self.char_reader.Get_Direction()
    return Checker.Check_Direct(self.angles[id], char_angle)

## Mapper

In [85]:
import pandas as pd
import numpy as np
import cv2
import pickle
from scipy.optimize import linear_sum_assignment
# from worker.character import Character
# from worker.checker import Checker
# from worker.drawer import Draw

class Map:
  def __init__(self) -> None:
    # self.name = None
    self.check_reader = Checker()
    self.char_reader = Character()
    # self.cam = None

  def Load_Map(self, cam: str) -> None:
    self.name = cam
    with open(f"./infor/map/{cam}.pkl", 'rb') as file:
      self.cam = pickle.load(file)
    return 

  def Active(self, num_img = 1800) -> None: # Khoảng 60s
    if self.cam.img_id >= num_img:
      return True
    return False

  def Save_Map(self) -> None:
    with open(f"./infor/map/{self.name}.pkl", 'wb') as file:
      pickle.dump(self.cam, file)
    return 


  #------------------------------------------MainOfMap
  '''
  char = [cls_id, *ft_color, *point, *speed, lost, violate]
  trong đó:
  ft_color = [ft_color_1, ft_color_2, ft_color_3]
  point = [x, y, w, h]
  speed = [spd_x, spd_y, spd_w, spd_h]
  len(char) = 14
  '''
  @staticmethod
  def Cal_Matrix(char_old: list, char_new: list):
    result = np.linalg.norm(np.array(char_old)[:, None, :] - np.array(char_new)[None, :, :], axis=2)
    return result

  def Re_ID(self, char_old: list, char_new: list) -> list:
    size_old = len(char_old)
    size_new = len(char_new)
    size = max(size_old, size_new)

    matrix = self.Cal_Matrix(char_old, char_new)
    matrix_padded = np.full((size, size), fill_value = 1e9)
    matrix_padded[:size_old, :size_new] = matrix

    old_ind, new_ind = linear_sum_assignment(matrix_padded)
    result = [[], []]
    for i, j in zip(old_ind, new_ind):
      if i < size_old and j < size_new and Map.Cal_Matrix([char_old[i]], [char_new[j]]) < 25:
        result[0].append(i)
        result[1].append(j)
    return result[0], result[1]

  def Re_Infor(self, char_old: list, char_new: list) -> None:
    if len(char_old) == 0:
      self.cam.chars = char_new
      return
    char_old = np.array(char_old)
    char_new = np.array(char_new)
    old_id, new_id = self.Re_ID(char_old[:, :8], char_new[:, :8])
    # Fix speed
    char_old[old_id, 8: 12] = char_new[new_id, 4: 8] - char_old[old_id, 4: 8]
    # Fix value
    char_old[old_id, :8] = char_new[new_id, :8]

    # add lost
    char_old[[i not in old_id for i in range(len(char_old))], 12] += 1
    char_old[[i in old_id for i in range(len(char_old))], 12] = 0
    # add new char
    char_new = char_new[[i not in new_id for i in range(len(char_new))]]
    if char_new.shape[0] != 0:
      char_old = np.vstack((char_old, char_new))
    self.cam.chars = char_old.tolist()
    return

  def Run(self, img , input: list, draw = 'None') -> None:
    self.cam.img_id += 1
    if len(input) == 0:
      return
    input = [Character.New_Char(ip) for ip in input]
    self.Re_Infor(self.cam.chars, input)
    for id in range(len(self.cam.chars)):
      self.char_reader.Read_Char(self.cam.chars[id])
      check_x, check_y = self.char_reader.Get_ID_Check()
      if len(self.cam.map[check_x][check_y]) == 0:
        self.cam.map[check_x][check_y] = Checker().Save()
      self.check_reader.Read_Checker(self.cam.map[check_x][check_y])
      self.cam.chars[id] = self.check_reader.Run(self.cam.img_id, self.cam.chars[id])
      self.cam.map[check_x][check_y] = self.check_reader.Save()
    
    if draw == 'All':
      img = Draw().DrawFromMap(img, self.cam.chars)
      cv2.imwrite(f".\infor\img\{self.name}.jpg", img)
    return

## Manager

In [ ]:
# from worker.map import Map

from dataclasses import dataclass, field
from typing import List
import pickle

@dataclass
class Map_Infor:
    img_id: int = 0
    map: List[List[List[List[int]]]] = field(default_factory = lambda: [[[] for _ in range(16)] for _ in range(16)])
    chars: List[List[int]] = field(default_factory = lambda: [])

class Manager:
    def __init__(self) -> None:
        self.map = Map()

    def run(self, img, cam: str, results: list, draw = 'None') -> None:
        while True:
            try:
                self.map.Load_Map(cam)
                break
            except:
                temp = Map_Infor()
                with open(f"./infor/map/{cam}.pkl", 'wb') as file:
                    pickle.dump(temp, file)
        
        self.map.Run(img, results, draw)
        self.map.Save_Map()


## Processer

In [87]:
import numpy as np
import cv2

class Process:

    @staticmethod
    def Extract_HSV(img, box):
        h_img, w_img = img.shape[:2]
        x_min = max(0, int(box[0] - box[2] / 2))
        y_min = max(0, int(box[1] - box[3] / 2))
        x_max = min(w_img, int(box[0] + box[2] / 2))
        y_max = min(h_img, int(box[1] + box[3] / 2))
        h = np.mean(img[y_min:y_max, x_min:x_max, 0]) / 255. * 100
        s = np.mean(img[y_min:y_max, x_min:x_max, 1]) / 255. * 100
        v = np.mean(img[y_min:y_max, x_min:x_max, 2]) / 255. * 100
        return [h, s, v]
    
    @staticmethod
    def Add_HSV_2(img, results):
        cls_id = [r[0] for r in results]
        boxes = [r[1:] for r in results]
        img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        temp = [
            list(map(float, [cls*100000] + list(map(float, Process.Extract_HSV(img_hsv, box))) + list(map(float, box)))) 
            for cls, box in zip(cls_id, boxes)
        ]

        return temp

    @staticmethod
    def Add_HSV(imgs, results) -> list:
        pro_results = []
        
        for i in range(len(imgs)):
            cls_id = results[i].boxes.cls.cpu().numpy()
            boxes = results[i].boxes.xywh.cpu().numpy()
            img_hsv = cv2.cvtColor(imgs[i], cv2.COLOR_BGR2HSV)
            temp = [
                list(map(float, [cls*100000] + list(map(float, Process.Extract_HSV(img_hsv, box))) + list(map(float, box)))) 
                for cls, box in zip(cls_id, boxes)
            ]
            pro_results.append(temp)

        return pro_results

# Thử Nghiệm

In [88]:
import time

path = ".\cam_test\lb"
path_2 = ".\cam_test\cam-1"
manager = Manager()
processer = Process()
cam = 'cam-1'
for id in range(37):
    with open(f"{path}\cam-1_{id}.txt", 'r') as file:
        data = file.read().split('\n')
    result = []
    for d in data:
        temp = d.split(' ')
        try:
            result.append(list(map(float, temp)))
        except:
            pass
    img = cv2.imread(f"{path_2}\cam-1_{id}.jpg")
    result = processer.Add_HSV_2(img, result)
    manager.run(img, cam, result, 'All')
    time.sleep(1)

Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu
Chay
luu


In [12]:
import pickle
cam = 'cam-1'

with open(f"./infor/map/{cam}.pkl", 'rb') as file:
    data = pickle.load(file)

In [13]:
for d in data.chars:
    print(d[4:])

In [17]:
run = True
id = 0
for d in range(16):
    for t in range(16):
        if len(data.map[d][t]) != 0:
            p_1 = (int(d * 53.25), int(t * 30))
            p_2 = (int((d + 1) * 53.25), int((t + 1) * 30))
            cv2.putText(img, str(id), (p_1[0] + 20, p_1[1] + 15),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            cv2.rectangle(img, p_1, p_2, color = (0, 255, 255), thickness=2)
            print(id, data.map[d][t])
            id += 1
cv2.imwrite(f'.\infor\img\cam-1.jpg', img)

True

In [18]:
for d in data.chars:
    print(d[8:])